## Differential Heatmaps: 2024 minus 2022

Analysis notebook accompanying the article **"Exploring EIA 930: The Shape of U.S. Electricity Demand"**.

Computes weather-normalized demand change (2024 vs 2022) by hour of day × day of year — a 24 × 365 matrix where each cell is the mean percentage change at that hour × day-of-year combination. February 29 is dropped from 2024 to align both years to a 365-day calendar.

Three region groups are shown, matching the heatmap figures in the article:

| Group | Entities |
|-------|----------|
| NY — NYIS Zones | Zones E, B, C, G, F (5 subregions) |
| NE — ISO New England | Zone 3 — Connecticut (ISNE-4003) |
| CAL — CAISO | PG&E, SCE, SDG&E (3 subregions) |

**Data requirement**: weather-normalized hourly demand parquet — update `NORM_SUB_PATH` in Cell 1 to your local path.


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import calendar as cal
from plotly.subplots import make_subplots

# Data: LBNL/EIA Cleaned Hourly Electricity Demand, Release 2 (2020-2024), weather-normalized.
# Update this path to point to your local weather-normalized subregions parquet.
NORM_SUB_PATH = '/Users/ashreeta/Downloads/Articles/LDD/weather_norm/data/hourly_normalized_subregions.parquet'

_SUBREGION_PARENTS = {
    'CISO': ['CISO-PGAE','CISO-SCE','CISO-SDGE','CISO-VEA'],
    'ERCO': ['ERCO-COAS','ERCO-EAST','ERCO-FWES','ERCO-NCEN','ERCO-NRTH','ERCO-SCEN','ERCO-SOUT','ERCO-WEST'],
    'ISNE': ['ISNE-4001','ISNE-4002','ISNE-4003','ISNE-4004','ISNE-4005','ISNE-4006','ISNE-4007','ISNE-4008'],
    'MISO': ['MISO-0001','MISO-0004','MISO-0006','MISO-0027','MISO-0035','MISO-8910'],
    'NYIS': ['NYIS-ZONA','NYIS-ZONB','NYIS-ZONC','NYIS-ZOND','NYIS-ZONE','NYIS-ZONF','NYIS-ZONG','NYIS-ZONH','NYIS-ZONI','NYIS-ZONJ','NYIS-ZONK'],
    'PJM':  ['PJM-AE','PJM-AEP','PJM-AP','PJM-ATSI','PJM-BC','PJM-CE','PJM-DAY','PJM-DEOK','PJM-DOM','PJM-DPL','PJM-DUQ','PJM-EKPC','PJM-JC','PJM-ME','PJM-PE','PJM-PEP','PJM-PL','PJM-PN','PJM-PS'],
    'SWPP': ['SWPP-CSWS','SWPP-EDE','SWPP-GRDA','SWPP-INDN','SWPP-KACY','SWPP-KCPL','SWPP-LES','SWPP-MPS','SWPP-NPPD','SWPP-OKGE','SWPP-OPPD','SWPP-SECI','SWPP-SPRM','SWPP-SPS','SWPP-WAUE','SWPP-WFEC','SWPP-WR'],
}

print("Loading subregion parquet...")
norm_sub = pd.read_parquet(NORM_SUB_PATH)
norm_sub['year'] = norm_sub['date_time'].dt.year

print("Aggregating to BA level...")
sub_present = set(norm_sub['entity'].unique())
parts = [norm_sub[~norm_sub['entity'].str.contains('-', regex=False)].copy()]
for ba, subs in _SUBREGION_PARENTS.items():
    in_data = [s for s in subs if s in sub_present]
    if not in_data:
        continue
    agg = (norm_sub[norm_sub['entity'].isin(in_data)]
           .groupby('date_time', as_index=False)['demand_mw_normalized']
           .sum(min_count=1))
    agg['entity'] = ba
    agg['year']   = agg['date_time'].dt.year
    parts.append(agg[['date_time','entity','demand_mw_normalized','year']])

norm_ba = pd.concat(parts, ignore_index=True)

norm_comb = pd.concat([
    norm_ba[~norm_ba['entity'].str.contains('-', regex=False)],
    norm_sub[norm_sub['entity'].str.contains('-', regex=False)],
], ignore_index=True)
norm_comb['year'] = norm_comb['date_time'].dt.year

print(f"Entities: {norm_comb['entity'].nunique()}")
print(f"Years: {sorted(norm_comb['year'].unique())}")


In [ ]:
# Entities grouped by region — each region gets its own colorscale
REGION_GROUPS = [
    ('NY — NYIS Zones', [
        ('NYIS-ZONE', 'Zone E  −6.6pp'),
        ('NYIS-ZONB', 'Zone B  −5.0pp'),
        ('NYIS-ZONC', 'Zone C  −5.2pp'),
        ('NYIS-ZONG', 'Zone G  −4.7pp'),
        ('NYIS-ZONF', 'Zone F  −3.9pp'),
    ]),
    ('NE — ISO New England', [
        ('ISNE-4003', 'Zone 3  −4.9pp'),
    ]),
    ('CAL — CAISO', [
        ('CISO-PGAE', 'PG&E  +4.9pp'),
        ('CISO-SCE',  'SCE   +6.1pp'),
        ('CISO-SDGE', 'SDG&E  +12.1pp'),
    ]),
]

# 365 ordered month-day keys (non-leap calendar)
MD365 = []
for m in range(1, 13):
    for d in range(1, cal.monthrange(2022, m)[1] + 1):
        MD365.append(f'{m:02d}-{d:02d}')

MTK_IDX  = [MD365.index(f'{m:02d}-01') for m in range(1, 13)]
MTK_TEXT = [cal.month_abbr[m] for m in range(1, 13)]

def build_matrix(df_ent, year):
    df_y = df_ent[df_ent['year'] == year].copy()
    df_y = df_y[~((df_y['date_time'].dt.month == 2) & (df_y['date_time'].dt.day == 29))]
    df_y['hour'] = df_y['date_time'].dt.hour
    df_y['md']   = df_y['date_time'].dt.strftime('%m-%d')
    return (df_y.pivot_table(index='md', columns='hour',
                             values='demand_mw_normalized', aggfunc='mean')
               .reindex(MD365))

# Pre-compute all diffs
all_diffs = {}
for _, ents in REGION_GROUPS:
    for ent, _ in ents:
        df_e = norm_comb[norm_comb['entity'] == ent]
        if df_e.empty:
            print(f'  WARNING: {ent} not found')
            continue
        m22 = build_matrix(df_e, 2022)
        m24 = build_matrix(df_e, 2024)
        d = (m24 - m22) / m22.abs() * 100
        if not d.isnull().all().all():
            all_diffs[ent] = d
            print(f'  {ent:<12} min={np.nanmin(d.values):+.1f}%  max={np.nanmax(d.values):+.1f}%')

# One figure per region
for region_label, ents in REGION_GROUPS:
    present = [(e, l) for e, l in ents if e in all_diffs]
    if not present:
        continue

    # Per-region symmetric colorscale
    reg_max = max(float(np.nanmax(np.abs(all_diffs[e].values))) for e, _ in present)
    cmax = float(np.ceil(reg_max / 5) * 5)

    ncols = len(present)
    fig = make_subplots(
        rows=1, cols=ncols,
        subplot_titles=[lbl for _, lbl in present],
        horizontal_spacing=0.06,
    )

    for i, (ent, lbl) in enumerate(present):
        show_cb = (i == len(present) - 1)
        fig.add_trace(go.Heatmap(
            z=all_diffs[ent].values,
            x=list(range(24)),
            y=list(range(365)),
            colorscale='RdBu_r',
            zmid=0,
            zmin=-cmax,
            zmax=cmax,
            showscale=show_cb,
            colorbar=dict(
                title=dict(text='% chg', font=dict(size=9)),
                thickness=12, tickfont=dict(size=8), ticksuffix='%',
            ) if show_cb else {},
            hovertemplate='Hour %{x}h, Day %{y}<br>% change: %{z:.1f}%<extra>' + lbl + '</extra>',
        ), row=1, col=i+1)

        fig.update_xaxes(
            tickvals=[0, 6, 12, 18, 23],
            ticktext=['0h', '6h', '12h', '18h', '23h'],
            tickfont_size=8, row=1, col=i+1,
        )
        fig.update_yaxes(
            tickvals=MTK_IDX, ticktext=MTK_TEXT,
            tickfont_size=8, autorange='reversed',
            row=1, col=i+1,
        )

    for ann in fig.layout.annotations:
        ann.font.size = 10

    fig.update_layout(
        title=(
            f'{region_label} — differential heatmaps 2024 vs 2022 (% change)<br>'
            f'<sup>Red = higher demand in 2024 · Blue = lower · '
            f'colorscale +/-{int(cmax)}%</sup>'
        ),
        height=420,
        template='plotly_white',
        margin=dict(t=100, l=50, r=70, b=30),
    )
    fig.show()
